<a href="https://colab.research.google.com/github/bibekgurau/CN6005/blob/main/week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch

In [2]:
from transformers import pipeline

# Load model (downloads first time only)
llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

print("Model loaded successfully")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model loaded successfully


In [3]:
from transformers import pipeline
# Load model (downloads first time only)
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct" #"microsoft/Phi-3-mini-4k-instruct"
)

print("Model loaded successfully")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded successfully


In [4]:
print("Before")
prompt = """
Answer the question in one sentence.

Question: What is Python?

Answer:
"""

result = llm(
    prompt,
    max_new_tokens=30,
    do_sample=False,
    return_full_text=False
)

print(result[0]["generated_text"])
print("after")

Before
Python is a high-level, interpreted programming language known for its simplicity and readability. It's widely used for web development, data analysis, artificial intelligence,
after


In [5]:
faq = {
    "deadline": "The submission deadline is 15 December.",
    "weight": "The assessment is worth 50% of the module.",
    "referencing": "The required referencing style is Harvard.",
    "module": "The module is AI and Emerging Technologies."
}

# --------------------------------------------------
# STEP : Search Function
# --------------------------------------------------

def search_knowledge(question):

    q = question.lower()

    if "deadline" in q:
        return faq["deadline"]

    elif "weight" in q or "mark" in q:
        return faq["weight"]

    elif "reference" in q or "referencing" in q:
        return faq["referencing"]

    elif "module" in q:
        return faq["module"]

    else:
        return None

# --------------------------------------------------
# STEP : Agent Function
# --------------------------------------------------
def agent(question):

    information = search_knowledge(question)

    print("Step 1: Search completed")
    print("Found:", information)

    if information is None:
        return "Sorry, I could not find that information."

    prompt = f"""
Answer the student's question using only the information provided.

Question:
{question}

Information:
{information}

Answer:
"""

    print("Step 2: Calling LLM")

    result = llm(
        prompt,
        max_new_tokens=30,
        do_sample=False,
        return_full_text=False
    )

    print("Step 3: Response received")

    return result[0]["generated_text"].strip()
# --------------------------------------------------
# STEP : Conversation Loop
# --------------------------------------------------

print("\nFAQ Agent Ready")
print("Type 'exit' to quit.\n")

while True:

    question = input("Student: ")

    if question.lower() == "exit":
        break

    answer = agent(question)

    print("\nAgent:", answer)
    print()


FAQ Agent Ready
Type 'exit' to quit.

Student: exit


In [6]:
# STEP 1: IMPORT LIBRARIES AND LOAD QWEN

import json
from datetime import date
from transformers import pipeline, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

llm = pipeline(
    "text-generation",
    model=MODEL,
    tokenizer=tokenizer
)

def generate_response(prompt, max_tokens=150):

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    result = llm(
        formatted_prompt,
        max_new_tokens=max_tokens,
        do_sample=False,
        return_full_text=False
    )

    return result[0]["generated_text"].strip()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [7]:
# STEP 2: CREATE THE KNOWLEDGE BASE

knowledge_base = {
    "module": "AI and Emerging Technologies",
    "deadline": "2026-12-15",
    "weighting": "50% of the module",
    "referencing": "Harvard referencing",
    "assessment": "Individual assignment"
}

In [8]:
# TOOL 1: SEARCH MODULE INFORMATION

def search_module_info(args, results):

    return knowledge_base
# TOOL 2: CALCULATE DAYS UNTIL DEADLINE

def calculate_days(args, results):

    deadline = date.fromisoformat(
        knowledge_base["deadline"]
    )

    today = date.today()
    remaining = (deadline - today).days

    return {
        "deadline": str(deadline),
        "days_remaining": remaining
    }
# TOOL 3: CREATE A SIMPLE STUDY PLAN

def create_study_plan(args, results):

    days = results.get("calculate_days", {})
    remaining = days.get("days_remaining")

    plan = [
        "Review the assignment brief and marking criteria.",
        "Identify the key topics and research requirements.",
        "Gather academic sources and relevant evidence.",
        "Develop an outline and organise your ideas.",
        "Write the assignment and review your work.",
        "Check Harvard referencing and submit before the deadline."
    ]

    if remaining is not None and remaining <= 7:
        plan.insert(0, "Prioritise completing and reviewing the assignment.")

    return plan

In [9]:
# STEP 4: CREATE THE TOOL REGISTRY

tools = {
    "search_module_info": search_module_info,
    "calculate_days": calculate_days,
    "create_study_plan": create_study_plan
}

tool_descriptions = """
Available tools:

1. search_module_info:
   Retrieves assignment and module information.

2. calculate_days:
   Calculates the number of days until the assignment deadline.

3. create_study_plan:
   Creates a study plan using available assignment information.
"""

In [10]:
# STEP 5: LLM-BASED PLANNING AND JSON VALIDATION

def create_plan(question, history):

    prompt = f"""
You are the planning component of a student support AI agent.

Understand the student's request and select the necessary tools.

{tool_descriptions}

Rules:
- Return a maximum of three steps.
- Use only the available tools.
- Search for information before creating a study plan.
- Calculate days when the student asks about the deadline.
- Use tools in the correct order.
- If no tool is needed, return an empty steps list.
- Return ONLY valid JSON. Do not include explanations.

Required JSON format:
{{
    "steps": [
        {{"tool": "tool_name", "args": {{}}}}
    ]
}}

Recent conversation:
{history}

Student request:
{question}
"""

    response = generate_response(prompt, max_tokens=150)

    # Extract the JSON object from the response
    try:
        start = response.find("{")
        end = response.rfind("}") + 1

        if start == -1 or end == 0:
            return None

        plan = json.loads(response[start:end])

        # Check the plan structure
        if not isinstance(plan, dict):
            return None

        steps = plan.get("steps")

        if not isinstance(steps, list) or len(steps) > 3:
            return None

        # Check every requested tool
        for step in steps:

            if not isinstance(step, dict):
                return None

            if step.get("tool") not in tools:
                return None

            if not isinstance(step.get("args"), dict):
                return None

        return plan

    except (json.JSONDecodeError, TypeError):
        return None

In [11]:
# STEP 6: EXECUTE TOOLS AND GENERATE RESPONSE

def execute_plan(plan):

    results = {}

    for step in plan["steps"]:

        tool_name = step["tool"]
        arguments = step["args"]

        print("Executing tool:", tool_name)

        tool = tools[tool_name]

        results[tool_name] = tool(arguments, results)

    return results


def generate_final_answer(question, results, history):

    prompt = f"""
You are a helpful university student support assistant.

Answer the student's question using the available information.

Do not invent assignment details.
If information is unavailable, say so clearly.
Explain the results in simple language.

Recent conversation:
{history}

Student question:
{question}

Information retrieved by the tools:
{json.dumps(results, indent=2)}

Provide a clear and helpful answer:
"""

    return generate_response(prompt, max_tokens=200)

In [13]:
# STEP 7: CONVERSATION MEMORY AND INTERACTION LOOP

history = []

def agent(question):

    # Create a plan
    plan = create_plan(question, history)

    if plan is None:
        return "I could not create a valid plan. Please try again."

    # Execute the approved tools
    results = execute_plan(plan)

    # Generate the final answer
    answer = generate_final_answer(
        question, results, history
    )

    # Store the interaction in memory
    history.append({
        "student": question,
        "agent": answer
    })

    # Retain only the last four interactions
    del history[:-4]

    return answer


print("====================================")
print("STUDENT SUPPORT AI AGENT")
print("====================================")
print("Type 'exit' to quit.\n")

while True:

    question = input("Student: ")

    if question.lower() == "exit":
        break

    answer = agent(question)

    print("\nAgent:", answer)
    print("\n------------------------------------")

STUDENT SUPPORT AI AGENT
Type 'exit' to quit.

Student: exit
